# pandas 3/3 : `groupby`, `merge` et `concat`

**Objectifs** : calculer des statistiques **par groupe** avec `groupby`, puis combiner des tables avec `merge` (jointure) et `concat` (empilement).

*Durée estimée : 45 min.*

In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/titanic.csv")
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 1. `groupby` : séparer, appliquer, combiner

Le principe (*split-apply-combine*) :

1. **séparer** les lignes en groupes selon les valeurs d'une colonne
2. **appliquer** une fonction (moyenne, somme, comptage…) à chaque groupe
3. **combiner** les résultats dans un nouveau tableau

In [20]:
df.groupby("Sex")["Survived"].mean()    # taux de survie par sexe

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [21]:
df.groupby("Pclass")["Age"].mean()      # âge moyen par classe

Pclass
1    38.233441
2    29.877630
3    25.140620
Name: Age, dtype: float64

On peut voir concrètement les groupes en les parcourant :

In [22]:
for classe, sous_df in df.groupby("Pclass"):
    print(f"Classe {classe} : {len(sous_df)} passagers, tarif moyen {sous_df['Fare'].mean():.1f}")

Classe 1 : 216 passagers, tarif moyen 84.2
Classe 2 : 184 passagers, tarif moyen 20.7
Classe 3 : 491 passagers, tarif moyen 13.7


### Plusieurs agrégations à la fois : `agg`

In [23]:
df.groupby("Pclass")["Fare"].agg(["count", "mean", "median", "max"])

,count,mean,median,max
Pclass,,,,
1,216,84.154687,60.2875,512.3292
2,184,20.662183,14.2500,73.5000
3,491,13.675550,8.0500,69.5500


Avec des **agrégations nommées**, on contrôle le nom des colonnes de résultat et on peut agréger plusieurs colonnes différentes :

In [24]:
df.groupby("Pclass").agg(
    nb_passagers=("PassengerId", "count"),
    age_moyen=("Age", "mean"),
    tarif_moyen=("Fare", "mean"),
    taux_survie=("Survived", "mean"),
)

,nb_passagers,age_moyen,tarif_moyen,taux_survie
Pclass,,,,
1,216,38.233441,84.154687,0.629630
2,184,29.877630,20.662183,0.472826
3,491,25.140620,13.675550,0.242363


### Grouper par plusieurs colonnes

In [25]:
res = df.groupby(["Pclass", "Sex"])["Survived"].mean()
res

Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

Le résultat a un **index à deux niveaux**. `unstack()` fait pivoter le dernier niveau en colonnes, ce qui donne un tableau plus lisible :

In [26]:
res.unstack()

Sex,female,male
Pclass,,
1,0.968085,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


In [27]:
# Même résultat avec pivot_table, souvent plus direct
df.pivot_table(index="Pclass", columns="Sex", values="Survived", aggfunc="mean")

Sex,female,male
Pclass,,
1,0.968085,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


### Récupérer un DataFrame « normal »

`reset_index()` transforme l'index de groupes en colonnes ordinaires (pratique pour la suite, notamment pour les graphiques).

In [28]:
df.groupby("Embarked")["Survived"].mean().reset_index()

,Embarked,Survived
0,C,0.553571
1,Q,0.389610
2,S,0.336957


In [29]:
# Et on peut trier le résultat
df.groupby("Embarked")["Fare"].mean().sort_values(ascending=False)

Embarked
C    59.954144
S    27.079812
Q    13.276030
Name: Fare, dtype: float64

---
### ✏️ Exercices (`groupby`)

### ✏️ Exercice 1

Calculez le taux de survie selon le port d'embarquement (`Embarked`).

In [30]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.groupby("Embarked")["Survived"].mean()
```

</details>

### ✏️ Exercice 2

Calculez l'âge moyen par classe **et** par sexe, sous forme d'un tableau avec les classes en lignes et les sexes en colonnes.

In [31]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.pivot_table(index="Pclass", columns="Sex", values="Age", aggfunc="mean")
```

</details>

### ✏️ Exercice 3

Pour chaque classe, donnez le nombre de passagers, le tarif minimum et le tarif maximum (agrégations nommées).

In [32]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.groupby("Pclass").agg(n=("PassengerId", "count"), fare_min=("Fare", "min"), fare_max=("Fare", "max"))
```

</details>

## 2. `merge` : joindre deux tables

On a souvent des informations réparties dans plusieurs tables, reliées par une **clé** commune. Ici, `Embarked` contient un code de port (`C`, `Q`, `S`) et un autre fichier décrit ces ports :

In [33]:
ports = pd.read_csv("../data/ports.csv")
ports

,Embarked,Port,Pays
0,C,Cherbourg,France
1,Q,Queenstown,Irlande
2,S,Southampton,Royaume-Uni
3,B,Belfast,Royaume-Uni


In [34]:
df_ports = df.merge(ports, on="Embarked")
df_ports[["Name", "Embarked", "Port", "Pays"]].head()
df_ports

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Port,Pays
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Southampton,Royaume-Uni
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cherbourg,France
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Southampton,Royaume-Uni
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Southampton,Royaume-Uni
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Southampton,Royaume-Uni
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Southampton,Royaume-Uni
885,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Southampton,Royaume-Uni
886,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Southampton,Royaume-Uni
887,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Cherbourg,France


### Les types de jointure : `how`

- `inner` (défaut) : on ne garde que les clés présentes **dans les deux** tables
- `left` : on garde toutes les lignes de la table de gauche
- `right` : toutes celles de la table de droite
- `outer` : tout, des deux côtés

Observons sur de petites tables :

In [35]:
etudiants = pd.DataFrame({"id": [1, 2, 3], "nom": ["Alice", "Bob", "Chloé"]})
notes = pd.DataFrame({"id": [2, 3, 4], "note": [14, 9, 16]})

print("--- inner"); print(etudiants.merge(notes, on="id", how="inner"))
print("--- left");  print(etudiants.merge(notes, on="id", how="left"))
print("--- right"); print(etudiants.merge(notes, on="id", how="right"))
print("--- outer"); print(etudiants.merge(notes, on="id", how="outer"))

--- inner
   id    nom  note
0   2    Bob    14
1   3  Chloé     9
--- left
   id    nom  note
0   1  Alice   NaN
1   2    Bob  14.0
2   3  Chloé   9.0
--- right
   id    nom  note
0   2    Bob    14
1   3  Chloé     9
2   4    NaN    16
--- outer
   id    nom  note
0   1  Alice   NaN
1   2    Bob  14.0
2   3  Chloé   9.0
3   4    NaN  16.0


Et sur nos données : deux passagers n'ont pas de port renseigné (`NaN`), et le port `B` (Belfast) n'a aucun passager.

In [36]:
print("df           :", len(df))
print("inner        :", len(df.merge(ports, on="Embarked", how="inner")))
print("left         :", len(df.merge(ports, on="Embarked", how="left")))
print("outer        :", len(df.merge(ports, on="Embarked", how="outer")))

df           : 891
inner        : 889
left         : 891
outer        : 892


**Réflexe à avoir** : après un `merge`, vérifier le nombre de lignes. Pour enrichir une table sans perdre de lignes, on utilise en général `how="left"`.

### Clés de noms différents et `indicator`

Si la colonne clé n'a pas le même nom dans les deux tables : `left_on` / `right_on`. `indicator=True` ajoute une colonne qui indique d'où vient chaque ligne.

In [37]:
ports2 = ports.rename(columns={"Embarked": "code_port"})
res = df.merge(ports2, left_on="Embarked", right_on="code_port", how="outer", indicator=True)
res["_merge"].value_counts()
res

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,code_port,Port,Pays,_merge
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B,Belfast,Royaume-Uni,right_only
1,2.0,1.0,1.0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC 17599,71.2833,C85,C,C,Cherbourg,France,both
2,10.0,1.0,2.0,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1.0,0.0,237736,30.0708,NaN,C,C,Cherbourg,France,both
3,20.0,1.0,3.0,"Masselmani, Mrs. Fatima",female,NaN,0.0,0.0,2649,7.2250,NaN,C,C,Cherbourg,France,both
4,27.0,0.0,3.0,"Emir, Mr. Farred Chehab",male,NaN,0.0,0.0,2631,7.2250,NaN,C,C,Cherbourg,France,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,887.0,0.0,2.0,"Montvila, Rev. Juozas",male,27.0,0.0,0.0,211536,13.0000,NaN,S,S,Southampton,Royaume-Uni,both
888,888.0,1.0,1.0,"Graham, Miss. Margaret Edith",female,19.0,0.0,0.0,112053,30.0000,B42,S,S,Southampton,Royaume-Uni,both
889,889.0,0.0,3.0,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1.0,2.0,W./C. 6607,23.4500,NaN,S,S,Southampton,Royaume-Uni,both
890,62.0,1.0,1.0,"Icard, Miss. Amelie",female,38.0,0.0,0.0,113572,80.0000,B28,NaN,NaN,NaN,NaN,left_only


### Exploiter la jointure : combiner merge et groupby

In [38]:
df.merge(ports, on="Embarked", how="left").groupby("Pays")["Survived"].agg(["count", "mean"])

,count,mean
Pays,,
France,168,0.553571
Irlande,77,0.389610
Royaume-Uni,644,0.336957


---
### ✏️ Exercices (`merge`)

### ✏️ Exercice 4

Quel est le tarif moyen payé par pays d'embarquement ? (jointure avec `ports`, puis `groupby`)

In [39]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.merge(ports, on="Embarked", how="left").groupby("Pays")["Fare"].mean()
```

</details>

### ✏️ Exercice 5

On dispose d'une table des classes : `classes = pd.DataFrame({"Pclass": [1, 2, 3], "Libelle": ["Première", "Seconde", "Troisième"]})`. Ajoutez le libellé à `df` et vérifiez que le nombre de lignes n'a pas changé.

In [40]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
classes = pd.DataFrame({"Pclass": [1, 2, 3], "Libelle": ["Première", "Seconde", "Troisième"]})
d = df.merge(classes, on="Pclass", how="left")
print(len(d) == len(df))
print(d[["Name", "Libelle"]].head())
```

</details>

## 3. `concat` : empiler des tables

`pd.concat` colle des tables **les unes sous les autres** (`axis=0`, défaut) ou **côte à côte** (`axis=1`). Cas typique : plusieurs fichiers de même structure (un par mois, par année…) à regrouper.

In [41]:
jan = pd.DataFrame({"produit": ["A", "B"], "ventes": [10, 20]})
fev = pd.DataFrame({"produit": ["A", "B"], "ventes": [12, 18]})

pd.concat([jan, fev])

,produit,ventes
0,A,10
1,B,20
0,A,12
1,B,18


L'index d'origine est conservé (d'où les doublons `0, 1, 0, 1`). Avec `ignore_index=True`, on renumérote ; avec `keys`, on garde l'origine de chaque bloc :

In [42]:
print(pd.concat([jan, fev], ignore_index=True))
print()
print(pd.concat([jan, fev], keys=["janvier", "février"]))

  produit  ventes
0       A      10
1       B      20
2       A      12
3       B      18

          produit  ventes
janvier 0       A      10
        1       B      20
février 0       A      12
        1       B      18


### Colonnes différentes

Les colonnes sont alignées **par nom** ; ce qui manque est rempli par `NaN`.

In [43]:
a = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
b = pd.DataFrame({"x": [5], "z": [6]})
pd.concat([a, b], ignore_index=True)

,x,y,z
0,1,3.0,NaN
1,2,4.0,NaN
2,5,NaN,6.0


### Côte à côte : `axis=1`

Les lignes sont alignées **par index**.

In [44]:
ages = df["Age"].head(3)
noms = df["Name"].head(3)
pd.concat([noms, ages], axis=1)

,Name,Age
0,"Braund, Mr. Owen Harris",22.0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
2,"Heikkinen, Miss. Laina",26.0


### Exemple complet : séparer puis recombiner

In [45]:
hommes = df[df["Sex"] == "male"]
femmes = df[df["Sex"] == "female"]

tout = pd.concat([hommes, femmes])
print(hommes.shape, femmes.shape, tout.shape)
print(tout.sort_index().equals(df))

(577, 12) (314, 12) (891, 12)
True


## Récapitulatif

| Je veux… | Code |
|---|---|
| statistique par groupe | `df.groupby("col")["val"].mean()` |
| plusieurs statistiques | `.agg(["mean", "max"])` ou `.agg(nom=("col", "fonction"))` |
| tableau croisé | `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)` |
| joindre deux tables | `df1.merge(df2, on="clé", how="left")` |
| empiler des lignes | `pd.concat([df1, df2], ignore_index=True)` |
| coller des colonnes | `pd.concat([df1, df2], axis=1)` |

---
## Exercices (`concat`)

### ✏️ Exercice 6

Découpez `df` en trois DataFrames selon la classe (`Pclass` = 1, 2, 3), puis recombinez-les avec `concat`. Vérifiez que le nombre de lignes est identique à celui de `df`.

In [18]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
c1 = df[df["Pclass"] == 1]
c2 = df[df["Pclass"] == 2]
c3 = df[df["Pclass"] == 3]
recombine = pd.concat([c1, c2, c3], ignore_index=True)
print(len(recombine) == len(df))
```

</details>

### ✏️ Exercice 7

**Défi** : pour chaque classe, trouvez les 2 passagers ayant payé le tarif le plus élevé.

In [8]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.sort_values("Fare", ascending=False).groupby("Pclass").head(2).sort_values(["Pclass", "Fare"], ascending=[True, False])[["Pclass", "Name", "Fare"]]
```

</details>